In [ ]:
# %% [markdown]
# # Ein-Box-Simulation mit STEP/STP-Datei
#
# Dieses Notebook:
#
# - lädt eine STEP/STP-Datei,
# - konvertiert sie nach STL,
# - liest Artikelmaße und Volumen aus,
# - definiert genau eine Box,
# - erlaubt die Anpassung der Simulationsparameter per Widgets,
# - startet die Simulation optional mit PyBullet-GUI,
# - gibt Packdichte, Box-Auslastung und Füllhöhe aus.

# %%
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if (PROJECT_ROOT / "backend").exists():
    BACKEND_ROOT = PROJECT_ROOT / "backend"
elif PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "backend").exists():
    BACKEND_ROOT = PROJECT_ROOT.parent / "backend"
else:
    BACKEND_ROOT = PROJECT_ROOT

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

DATA_DIR = BACKEND_ROOT / "data"
SAMPLES_DIR = BACKEND_ROOT / "samples"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

print("Backend root:", BACKEND_ROOT)
print("Data dir:", DATA_DIR)
print("Samples dir:", SAMPLES_DIR)

# %%
from app.simulation.sim import SimulationConfig, PackagingSimulation
from app.packing.models import Item, Box

import inspect
import pandas as pd
import ipywidgets as widgets

from IPython.display import display, clear_output

try:
    import cadquery as cq
    CADQUERY_AVAILABLE = True
except Exception as exc:
    CADQUERY_AVAILABLE = False
    print("CadQuery konnte nicht importiert werden:", exc)

print("CadQuery verfügbar:", CADQUERY_AVAILABLE)

# %%
def resolve_input_path(path_value):
    path = Path(str(path_value)).expanduser()

    if path.is_absolute() and path.exists():
        return path.resolve()

    candidates = [
        BACKEND_ROOT / path,
        DATA_DIR / path,
        SAMPLES_DIR / path,
        Path.cwd() / path,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(f"Datei nicht gefunden: {path_value}")


def load_step_workplane(step_path):
    if not CADQUERY_AVAILABLE:
        raise RuntimeError("CadQuery ist nicht verfügbar. Bitte cadquery installieren.")

    return cq.importers.importStep(str(step_path))


def get_step_dimensions_and_volume(step_path):
    workplane = load_step_workplane(step_path)
    shapes = list(workplane.vals())

    if not shapes:
        raise ValueError("Die STEP/STP-Datei enthält keine lesbaren Shapes.")

    compound = cq.Compound.makeCompound(shapes)
    bbox = compound.BoundingBox()

    volume = 0.0
    for shape in shapes:
        try:
            volume += float(shape.Volume())
        except Exception:
            pass

    return {
        "length": float(bbox.xlen),
        "width": float(bbox.ylen),
        "height": float(bbox.zlen),
        "volume": float(volume),
    }


def convert_step_to_stl(step_path, stl_path):
    workplane = load_step_workplane(step_path)
    stl_path = Path(stl_path)
    stl_path.parent.mkdir(parents=True, exist_ok=True)

    cq.exporters.export(workplane, str(stl_path))

    return stl_path.resolve()


def instantiate_dataclass_or_model(model_cls, **values):
    signature = inspect.signature(model_cls)
    accepted = {
        key: value
        for key, value in values.items()
        if key in signature.parameters
    }
    return model_cls(**accepted)


def make_item_from_dimensions(dimensions):
    return instantiate_dataclass_or_model(
        Item,
        length=dimensions["length"],
        width=dimensions["width"],
        height=dimensions["height"],
        name="STEP-Artikel",
        weight=0.0,
    )


def make_single_box(name, length, width, height, capacity_lhm=1):
    return instantiate_dataclass_or_model(
        Box,
        name=name,
        length=length,
        width=width,
        height=height,
        capacityLHM=capacity_lhm,
        lhm_capacity=capacity_lhm,
    )


def build_simulation_config(
    *,
    item,
    quantity,
    box,
    stl_file,
    collision_file,
    mesh_volume,
    item_mass,
    mesh_scale,
    wall_thickness,
    fixed_time_step,
    solver_iterations,
    max_simulation_steps,
    min_simulation_steps,
    settle_check_interval,
    height_change_mm,
    settle_duration,
    settle_force_scale,
    settle_frequency,
    fit_height_tolerance,
    random_seed,
    runs_per_box,
    use_gui,
    parallel_simulations,
):
    return SimulationConfig(
        item=item,
        item_quantity=quantity,
        boxes=[box],
        stl_file=str(stl_file),
        collision_file=collision_file,
        mesh_volume=mesh_volume,
        item_mass=item_mass,
        mesh_scale=mesh_scale,
        wall_thickness=wall_thickness,
        fixed_time_step=fixed_time_step,
        solver_iterations=solver_iterations,
        max_simulation_steps=max_simulation_steps,
        min_simulation_steps=min_simulation_steps,
        settle_check_interval=settle_check_interval,
        height_change_mm=height_change_mm,
        settle_duration=settle_duration,
        settle_force_scale=settle_force_scale,
        settle_frequency=settle_frequency,
        fit_height_tolerance=fit_height_tolerance,
        random_seed=random_seed,
        runs_per_box=runs_per_box,
        use_gui=use_gui,
        parallel_simulations=parallel_simulations,
    )


def result_to_dataframe(result):
    results = result.get("results", []) if isinstance(result, dict) else []

    rows = []
    for index, entry in enumerate(results, start=1):
        box = entry.get("box", {})
        rows.append(
            {
                "rank": index,
                "box_name": box.get("name"),
                "box_length_mm": box.get("length"),
                "box_width_mm": box.get("width"),
                "box_height_mm": box.get("height"),
                "filling_height_mm": entry.get("filling_height_mm"),
                "packing_density_percent": entry.get("packing_density_percent"),
                "box_utilization_percent": entry.get("box_utilization_percent"),
                "fits_in_box": entry.get("fits_in_box"),
                "articles_per_lhm": entry.get("articles_per_lhm"),
            }
        )

    return pd.DataFrame(rows)


def print_result_summary(result):
    results = result.get("results", []) if isinstance(result, dict) else []

    if not results:
        print("Keine passende Box gefunden oder keine Ergebnisse vorhanden.")
        return pd.DataFrame()

    for index, entry in enumerate(results, start=1):
        box = entry["box"]

        print(f"Ergebnis {index}")
        print(f"  Box: {box.get('name')}")
        print(f"  Maße: {box.get('length')} x {box.get('width')} x {box.get('height')} mm")
        print(f"  Füllhöhe: {entry.get('filling_height_mm', 0):.2f} mm")
        print(f"  Packdichte: {entry.get('packing_density_percent', 0):.2f} %")
        print(f"  Box-Auslastung: {entry.get('box_utilization_percent', 0):.2f} %")
        print(f"  Passt in Box: {entry.get('fits_in_box')}")
        print(f"  Artikel pro LHM: {entry.get('articles_per_lhm')}")
        print()

    df = result_to_dataframe(result)
    display(df)

    best = max(results, key=lambda entry: entry.get("packing_density_percent", 0))

    print("Beste Packdichte:")
    print(f"  Box: {best['box'].get('name')}")
    print(f"  Packdichte: {best.get('packing_density_percent', 0):.2f} %")
    print(f"  Füllhöhe: {best.get('filling_height_mm', 0):.2f} mm")

    return df

# %%
default_step_files = sorted(SAMPLES_DIR.glob("*.stp")) + sorted(SAMPLES_DIR.glob("*.step"))
default_step_path = str(default_step_files[0]) if default_step_files else str(SAMPLES_DIR / "dein_artikel.stp")

step_file_widget = widgets.Text(
    value=default_step_path,
    description="STEP/STP",
    layout=widgets.Layout(width="900px"),
)

box_name_widget = widgets.Text(
    value="Simulationsbox",
    description="Box",
    layout=widgets.Layout(width="400px"),
)

box_length_widget = widgets.FloatText(value=220.0, description="Länge mm")
box_width_widget = widgets.FloatText(value=160.0, description="Breite mm")
box_height_widget = widgets.FloatText(value=140.0, description="Höhe mm")
capacity_lhm_widget = widgets.FloatText(value=1.0, description="capacityLHM")

quantity_widget = widgets.IntText(value=100, description="Menge")
item_mass_widget = widgets.FloatText(value=0.01, description="Masse kg")
mesh_volume_override_widget = widgets.FloatText(value=0.0, description="Vol. Override")

mesh_scale_x_widget = widgets.FloatText(value=0.001, description="Scale X")
mesh_scale_y_widget = widgets.FloatText(value=0.001, description="Scale Y")
mesh_scale_z_widget = widgets.FloatText(value=0.001, description="Scale Z")

wall_thickness_widget = widgets.FloatText(value=0.01, description="Wand m")
fixed_time_step_widget = widgets.FloatText(value=1 / 480, description="Time step")
solver_iterations_widget = widgets.IntText(value=80, description="Solver iter.")

max_simulation_steps_widget = widgets.IntText(value=350, description="Max steps")
min_simulation_steps_widget = widgets.IntText(value=60, description="Min steps")
settle_check_interval_widget = widgets.IntText(value=20, description="Check int.")

height_change_mm_widget = widgets.FloatText(value=0.5, description="Δ Höhe mm")
settle_duration_widget = widgets.FloatText(value=0.25, description="Settle s")
settle_force_scale_widget = widgets.FloatText(value=0.04, description="Force scale")
settle_frequency_widget = widgets.FloatText(value=10.0, description="Freq.")
fit_height_tolerance_widget = widgets.FloatText(value=0.02, description="Fit tol.")

random_seed_widget = widgets.IntText(value=42, description="Seed")
runs_per_box_widget = widgets.IntText(value=1, description="Runs")

use_gui_widget = widgets.Checkbox(value=False, description="PyBullet GUI verwenden")
parallel_simulations_widget = widgets.Checkbox(value=False, description="Parallel simulieren")
use_step_volume_widget = widgets.Checkbox(value=True, description="STEP-Volumen verwenden")

load_button = widgets.Button(description="STEP laden / STL erzeugen", button_style="info")
run_button = widgets.Button(description="Simulation starten", button_style="success")

output = widgets.Output()

ui = widgets.VBox(
    [
        widgets.HTML("<h3>Datei</h3>"),
        step_file_widget,
        load_button,
        widgets.HTML("<h3>Box</h3>"),
        widgets.HBox([box_name_widget, capacity_lhm_widget]),
        widgets.HBox([box_length_widget, box_width_widget, box_height_widget]),
        widgets.HTML("<h3>Artikel / Menge</h3>"),
        widgets.HBox([quantity_widget, item_mass_widget, mesh_volume_override_widget]),
        use_step_volume_widget,
        widgets.HTML("<h3>Mesh-Skalierung</h3>"),
        widgets.HBox([mesh_scale_x_widget, mesh_scale_y_widget, mesh_scale_z_widget]),
        widgets.HTML("<h3>Physik / Solver</h3>"),
        widgets.HBox([wall_thickness_widget, fixed_time_step_widget, solver_iterations_widget]),
        widgets.HBox([max_simulation_steps_widget, min_simulation_steps_widget, settle_check_interval_widget]),
        widgets.HBox([height_change_mm_widget, settle_duration_widget, fit_height_tolerance_widget]),
        widgets.HBox([settle_force_scale_widget, settle_frequency_widget]),
        widgets.HTML("<h3>Ausführung</h3>"),
        widgets.HBox([random_seed_widget, runs_per_box_widget]),
        widgets.HBox([use_gui_widget, parallel_simulations_widget]),
        run_button,
        output,
    ]
)

display(ui)

# %%
STATE = {
    "step_path": None,
    "stl_path": None,
    "dimensions": None,
    "item": None,
    "box": None,
    "config": None,
    "result": None,
    "df": None,
}


def on_load_step_clicked(_):
    with output:
        clear_output()

        try:
            step_path = resolve_input_path(step_file_widget.value)
            stl_path = DATA_DIR / "simulation_input.stl"

            print("Lade STEP/STP-Datei:")
            print(" ", step_path)

            dimensions = get_step_dimensions_and_volume(step_path)
            converted_stl_path = convert_step_to_stl(step_path, stl_path)

            STATE["step_path"] = step_path
            STATE["stl_path"] = converted_stl_path
            STATE["dimensions"] = dimensions
            STATE["item"] = make_item_from_dimensions(dimensions)

            print()
            print("STEP/STP erfolgreich geladen.")
            print("STL erzeugt:")
            print(" ", converted_stl_path)
            print()
            print("Artikelmaße aus Bounding Box:")
            print(f"  Länge: {dimensions['length']:.3f} mm")
            print(f"  Breite: {dimensions['width']:.3f} mm")
            print(f"  Höhe:  {dimensions['height']:.3f} mm")
            print(f"  Volumen: {dimensions['volume']:.3f} mm³")

        except Exception as exc:
            print("Fehler beim Laden/Konvertieren:")
            print(exc)


def on_run_clicked(_):
    with output:
        clear_output()

        try:
            if STATE["dimensions"] is None or STATE["stl_path"] is None:
                print("STEP/STP wurde noch nicht geladen. Lade Datei jetzt automatisch...")
                step_path = resolve_input_path(step_file_widget.value)
                stl_path = DATA_DIR / "simulation_input.stl"

                dimensions = get_step_dimensions_and_volume(step_path)
                converted_stl_path = convert_step_to_stl(step_path, stl_path)

                STATE["step_path"] = step_path
                STATE["stl_path"] = converted_stl_path
                STATE["dimensions"] = dimensions
                STATE["item"] = make_item_from_dimensions(dimensions)

            dimensions = STATE["dimensions"]
            item = STATE["item"]
            stl_path = STATE["stl_path"]

            box = make_single_box(
                name=box_name_widget.value,
                length=float(box_length_widget.value),
                width=float(box_width_widget.value),
                height=float(box_height_widget.value),
                capacity_lhm=float(capacity_lhm_widget.value),
            )

            if use_step_volume_widget.value:
                mesh_volume = float(dimensions["volume"])
            else:
                mesh_volume = float(mesh_volume_override_widget.value)

            config = build_simulation_config(
                item=item,
                quantity=int(quantity_widget.value),
                box=box,
                stl_file=stl_path,
                collision_file=None,
                mesh_volume=mesh_volume,
                item_mass=float(item_mass_widget.value),
                mesh_scale=(
                    float(mesh_scale_x_widget.value),
                    float(mesh_scale_y_widget.value),
                    float(mesh_scale_z_widget.value),
                ),
                wall_thickness=float(wall_thickness_widget.value),
                fixed_time_step=float(fixed_time_step_widget.value),
                solver_iterations=int(solver_iterations_widget.value),
                max_simulation_steps=int(max_simulation_steps_widget.value),
                min_simulation_steps=int(min_simulation_steps_widget.value),
                settle_check_interval=int(settle_check_interval_widget.value),
                height_change_mm=float(height_change_mm_widget.value),
                settle_duration=float(settle_duration_widget.value),
                settle_force_scale=float(settle_force_scale_widget.value),
                settle_frequency=float(settle_frequency_widget.value),
                fit_height_tolerance=float(fit_height_tolerance_widget.value),
                random_seed=int(random_seed_widget.value),
                runs_per_box=int(runs_per_box_widget.value),
                use_gui=bool(use_gui_widget.value),
                parallel_simulations=bool(parallel_simulations_widget.value),
            )

            STATE["box"] = box
            STATE["config"] = config

            print("Starte Simulation...")
            print()
            print("Artikel:")
            print(f"  Länge: {dimensions['length']:.3f} mm")
            print(f"  Breite: {dimensions['width']:.3f} mm")
            print(f"  Höhe:  {dimensions['height']:.3f} mm")
            print(f"  Volumen für Dichte: {mesh_volume:.3f} mm³")
            print()
            print("Box:")
            print(f"  Name: {box_name_widget.value}")
            print(f"  Länge: {box_length_widget.value:.3f} mm")
            print(f"  Breite: {box_width_widget.value:.3f} mm")
            print(f"  Höhe:  {box_height_widget.value:.3f} mm")
            print()
            print("Simulationsparameter:")
            print(f"  Menge: {quantity_widget.value}")
            print(f"  PyBullet GUI: {use_gui_widget.value}")
            print(f"  Parallel: {parallel_simulations_widget.value}")
            print(f"  Runs pro Box: {runs_per_box_widget.value}")
            print(f"  STL: {stl_path}")
            print()

            simulation = PackagingSimulation(config)
            result = simulation.run()

            STATE["result"] = result

            print("Simulation abgeschlossen.")
            print()

            df = print_result_summary(result)
            STATE["df"] = df

        except Exception as exc:
            print("Fehler während der Simulation:")
            print(type(exc).__name__, exc)


load_button.on_click(on_load_step_clicked)
run_button.on_click(on_run_clicked)

# %%
# Optional: STEP/STP ohne Button laden
#
# Diese Zelle ist praktisch, wenn du das Notebook linear ausführen willst.

try:
    step_path = resolve_input_path(step_file_widget.value)
    stl_path = DATA_DIR / "simulation_input.stl"

    dimensions = get_step_dimensions_and_volume(step_path)
    converted_stl_path = convert_step_to_stl(step_path, stl_path)

    STATE["step_path"] = step_path
    STATE["stl_path"] = converted_stl_path
    STATE["dimensions"] = dimensions
    STATE["item"] = make_item_from_dimensions(dimensions)

    print("STEP/STP geladen:")
    print(step_path)
    print()
    print("STL erzeugt:")
    print(converted_stl_path)
    print()
    print("Artikelmaße:")
    print(f"  Länge: {dimensions['length']:.3f} mm")
    print(f"  Breite: {dimensions['width']:.3f} mm")
    print(f"  Höhe:  {dimensions['height']:.3f} mm")
    print(f"  Volumen: {dimensions['volume']:.3f} mm³")

except Exception as exc:
    print("Automatisches Laden übersprungen/fehlgeschlagen:")
    print(type(exc).__name__, exc)

# %%
# Optional: Simulation ohne Button starten
#
# Diese Zelle nutzt die aktuellen Widget-Werte.

if STATE["dimensions"] is None or STATE["stl_path"] is None:
    raise RuntimeError("Bitte zuerst STEP/STP laden oder die vorherige Zelle ausführen.")

dimensions = STATE["dimensions"]
item = STATE["item"]
stl_path = STATE["stl_path"]

box = make_single_box(
    name=box_name_widget.value,
    length=float(box_length_widget.value),
    width=float(box_width_widget.value),
    height=float(box_height_widget.value),
    capacity_lhm=float(capacity_lhm_widget.value),
)

mesh_volume = (
    float(dimensions["volume"])
    if use_step_volume_widget.value
    else float(mesh_volume_override_widget.value)
)

config = build_simulation_config(
    item=item,
    quantity=int(quantity_widget.value),
    box=box,
    stl_file=stl_path,
    collision_file=None,
    mesh_volume=mesh_volume,
    item_mass=float(item_mass_widget.value),
    mesh_scale=(
        float(mesh_scale_x_widget.value),
        float(mesh_scale_y_widget.value),
        float(mesh_scale_z_widget.value),
    ),
    wall_thickness=float(wall_thickness_widget.value),
    fixed_time_step=float(fixed_time_step_widget.value),
    solver_iterations=int(solver_iterations_widget.value),
    max_simulation_steps=int(max_simulation_steps_widget.value),
    min_simulation_steps=int(min_simulation_steps_widget.value),
    settle_check_interval=int(settle_check_interval_widget.value),
    height_change_mm=float(height_change_mm_widget.value),
    settle_duration=float(settle_duration_widget.value),
    settle_force_scale=float(settle_force_scale_widget.value),
    settle_frequency=float(settle_frequency_widget.value),
    fit_height_tolerance=float(fit_height_tolerance_widget.value),
    random_seed=int(random_seed_widget.value),
    runs_per_box=int(runs_per_box_widget.value),
    use_gui=bool(use_gui_widget.value),
    parallel_simulations=bool(parallel_simulations_widget.value),
)

STATE["box"] = box
STATE["config"] = config

print("Starte Simulation...")
print()
print("Artikel:")
print(f"  Länge: {dimensions['length']:.3f} mm")
print(f"  Breite: {dimensions['width']:.3f} mm")
print(f"  Höhe:  {dimensions['height']:.3f} mm")
print(f"  Volumen für Dichte: {mesh_volume:.3f} mm³")
print()
print("Box:")
print(f"  Name: {box_name_widget.value}")
print(f"  Länge: {box_length_widget.value:.3f} mm")
print(f"  Breite: {box_width_widget.value:.3f} mm")
print(f"  Höhe:  {box_height_widget.value:.3f} mm")
print()
print("Simulationsparameter:")
print(f"  Menge: {quantity_widget.value}")
print(f"  PyBullet GUI: {use_gui_widget.value}")
print(f"  Parallel: {parallel_simulations_widget.value}")
print(f"  Runs pro Box: {runs_per_box_widget.value}")
print(f"  STL: {stl_path}")
print()

simulation = PackagingSimulation(config)
result = simulation.run()

STATE["result"] = result

print("Simulation abgeschlossen.")
print()

df = print_result_summary(result)
STATE["df"] = df

# %%
# Rohdaten anzeigen

STATE["result"]

# %%
# Ergebnis-DataFrame anzeigen

if STATE["df"] is not None:
    display(STATE["df"])
else:
    print("Noch kein DataFrame vorhanden. Bitte zuerst Simulation ausführen.")

# %%
# Beste Packdichte separat anzeigen

result = STATE["result"]

if not result or not result.get("results"):
    print("Keine Ergebnisse verfügbar.")
else:
    best = max(result["results"], key=lambda entry: entry.get("packing_density_percent", 0))

    print("Beste Packdichte:")
    print(f"  Box: {best['box'].get('name')}")
    print(f"  Maße: {best['box'].get('length')} x {best['box'].get('width')} x {best['box'].get('height')} mm")
    print(f"  Füllhöhe: {best.get('filling_height_mm', 0):.2f} mm")
    print(f"  Packdichte: {best.get('packing_density_percent', 0):.2f} %")
    print(f"  Box-Auslastung: {best.get('box_utilization_percent', 0):.2f} %")
    print(f"  Passt in Box: {best.get('fits_in_box')}")
    print(f"  Artikel pro LHM: {best.get('articles_per_lhm')}")

# %%
# Parameter-Sweep für eine einzelne Box
#
# Hier kannst du mehrere Mengen in derselben Box testen.

quantities_to_test = [10, 25, 50, 75, 100]

sweep_rows = []

if STATE["dimensions"] is None or STATE["stl_path"] is None:
    print("Bitte zuerst STEP/STP laden.")
else:
    dimensions = STATE["dimensions"]
    item = STATE["item"]
    stl_path = STATE["stl_path"]

    for quantity in quantities_to_test:
        box = make_single_box(
            name=box_name_widget.value,
            length=float(box_length_widget.value),
            width=float(box_width_widget.value),
            height=float(box_height_widget.value),
            capacity_lhm=float(capacity_lhm_widget.value),
        )

        mesh_volume = (
            float(dimensions["volume"])
            if use_step_volume_widget.value
            else float(mesh_volume_override_widget.value)
        )

        config = build_simulation_config(
            item=item,
            quantity=int(quantity),
            box=box,
            stl_file=stl_path,
            collision_file=None,
            mesh_volume=mesh_volume,
            item_mass=float(item_mass_widget.value),
            mesh_scale=(
                float(mesh_scale_x_widget.value),
                float(mesh_scale_y_widget.value),
                float(mesh_scale_z_widget.value),
            ),
            wall_thickness=float(wall_thickness_widget.value),
            fixed_time_step=float(fixed_time_step_widget.value),
            solver_iterations=int(solver_iterations_widget.value),
            max_simulation_steps=int(max_simulation_steps_widget.value),
            min_simulation_steps=int(min_simulation_steps_widget.value),
            settle_check_interval=int(settle_check_interval_widget.value),
            height_change_mm=float(height_change_mm_widget.value),
            settle_duration=float(settle_duration_widget.value),
            settle_force_scale=float(settle_force_scale_widget.value),
            settle_frequency=float(settle_frequency_widget.value),
            fit_height_tolerance=float(fit_height_tolerance_widget.value),
            random_seed=int(random_seed_widget.value),
            runs_per_box=int(runs_per_box_widget.value),
            use_gui=False,
            parallel_simulations=False,
        )

        print(f"Teste Menge {quantity}...")

        result = PackagingSimulation(config).run()
        results = result.get("results", [])

        if not results:
            sweep_rows.append(
                {
                    "quantity": quantity,
                    "fits_in_box": False,
                    "packing_density_percent": None,
                    "box_utilization_percent": None,
                    "filling_height_mm": None,
                    "articles_per_lhm": None,
                }
            )
            continue

        entry = results[0]

        sweep_rows.append(
            {
                "quantity": quantity,
                "fits_in_box": entry.get("fits_in_box"),
                "packing_density_percent": entry.get("packing_density_percent"),
                "box_utilization_percent": entry.get("box_utilization_percent"),
                "filling_height_mm": entry.get("filling_height_mm"),
                "articles_per_lhm": entry.get("articles_per_lhm"),
            }
        )

    sweep_df = pd.DataFrame(sweep_rows)
    display(sweep_df)

# %%
# Parameter-Sweep über Boxhöhen
#
# Damit kannst du evaluieren, welche Höhe für dieselbe Grundfläche sinnvoll ist.

box_heights_to_test = [80, 100, 120, 140, 160, 180, 200]

height_sweep_rows = []

if STATE["dimensions"] is None or STATE["stl_path"] is None:
    print("Bitte zuerst STEP/STP laden.")
else:
    dimensions = STATE["dimensions"]
    item = STATE["item"]
    stl_path = STATE["stl_path"]

    mesh_volume = (
        float(dimensions["volume"])
        if use_step_volume_widget.value
        else float(mesh_volume_override_widget.value)
    )

    for test_height in box_heights_to_test:
        box = make_single_box(
            name=f"{box_name_widget.value}_{test_height}mm",
            length=float(box_length_widget.value),
            width=float(box_width_widget.value),
            height=float(test_height),
            capacity_lhm=float(capacity_lhm_widget.value),
        )

        config = build_simulation_config(
            item=item,
            quantity=int(quantity_widget.value),
            box=box,
            stl_file=stl_path,
            collision_file="data/ausgabe_vhacd.obj",
            mesh_volume=mesh_volume,
            item_mass=float(item_mass_widget.value),
            mesh_scale=(
                float(mesh_scale_x_widget.value),
                float(mesh_scale_y_widget.value),
                float(mesh_scale_z_widget.value),
            ),
            wall_thickness=float(wall_thickness_widget.value),
            fixed_time_step=float(fixed_time_step_widget.value),
            solver_iterations=int(solver_iterations_widget.value),
            max_simulation_steps=int(max_simulation_steps_widget.value),
            min_simulation_steps=int(min_simulation_steps_widget.value),
            settle_check_interval=int(settle_check_interval_widget.value),
            height_change_mm=float(height_change_mm_widget.value),
            settle_duration=float(settle_duration_widget.value),
            settle_force_scale=float(settle_force_scale_widget.value),
            settle_frequency=float(settle_frequency_widget.value),
            fit_height_tolerance=float(fit_height_tolerance_widget.value),
            random_seed=int(random_seed_widget.value),
            runs_per_box=int(runs_per_box_widget.value),
            use_gui=False,
            parallel_simulations=False,
        )

        print(f"Teste Boxhöhe {test_height} mm...")

        result = PackagingSimulation(config).run()
        results = result.get("results", [])

        if not results:
            height_sweep_rows.append(
                {
                    "box_height_mm": test_height,
                    "fits_in_box": False,
                    "packing_density_percent": None,
                    "box_utilization_percent": None,
                    "filling_height_mm": None,
                    "articles_per_lhm": None,
                }
            )
            continue

        entry = results[0]

        height_sweep_rows.append(
            {
                "box_height_mm": test_height,
                "fits_in_box": entry.get("fits_in_box"),
                "packing_density_percent": entry.get("packing_density_percent"),
                "box_utilization_percent": entry.get("box_utilization_percent"),
                "filling_height_mm": entry.get("filling_height_mm"),
                "articles_per_lhm": entry.get("articles_per_lhm"),
            }
        )

    height_sweep_df = pd.DataFrame(height_sweep_rows)
    display(height_sweep_df)

Backend root: C:\Users\QJ095K\Desktop\BinPacking neu\test\backend
Data dir: C:\Users\QJ095K\Desktop\BinPacking neu\test\backend\data
Samples dir: C:\Users\QJ095K\Desktop\BinPacking neu\test\backend\samples
CadQuery verfügbar: True


STEP/STP geladen:
C:\Users\QJ095K\Desktop\BinPacking neu\test\backend\samples\pxc_1777280_07_FRONT-MSTB-2-5-2-ST-5-08_3D.stp

STL erzeugt:
C:\Users\QJ095K\Desktop\BinPacking neu\test\backend\data\simulation_input.stl

Artikelmaße:
  Länge: 10.170 mm
  Breite: 27.200 mm
  Höhe:  14.900 mm
  Volumen: 2378.102 mm³
Starte Simulation...

Artikel:
  Länge: 10.170 mm
  Breite: 27.200 mm
  Höhe:  14.900 mm
  Volumen für Dichte: 2378.102 mm³

Box:
  Name: Simulationsbox
  Länge: 220.000 mm
  Breite: 160.000 mm
  Höhe:  140.000 mm

Simulationsparameter:
  Menge: 100
  PyBullet GUI: False
  Parallel: False
  Runs pro Box: 1
  STL: C:\Users\QJ095K\Desktop\BinPacking neu\test\backend\data\simulation_input.stl

Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 29.83 mm
Relative Füllhöhe der Kiste: 21.31 %
Theoretisches Artikelvolumen gesamt: 237.81 cm^3
Beanspruchtes Box-Volumen: 1049.96 cm^3
Erreichte Fülldichte: 22.65 %
Simulation abgeschlossen.

Ergebnis 1


,rank,box_name,box_length_mm,box_width_mm,box_height_mm,filling_height_mm,packing_density_percent,box_utilization_percent,fits_in_box,articles_per_lhm
0,1,Simulationsbox,220.0,160.0,140.0,29.828368,22.649484,None,True,100.0


Beste Packdichte:
  Box: Simulationsbox
  Packdichte: 22.65 %
  Füllhöhe: 29.83 mm


,rank,box_name,box_length_mm,box_width_mm,box_height_mm,filling_height_mm,packing_density_percent,box_utilization_percent,fits_in_box,articles_per_lhm
0,1,Simulationsbox,220.0,160.0,140.0,29.828368,22.649484,None,True,100.0


Beste Packdichte:
  Box: Simulationsbox
  Maße: 220.0 x 160.0 x 140.0 mm
  Füllhöhe: 29.83 mm
  Packdichte: 22.65 %
  Box-Auslastung: 0.00 %
  Passt in Box: True
  Artikel pro LHM: 100.0
Teste Menge 10...
Spawne 10 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 29.74 mm
Relative Füllhöhe der Kiste: 21.24 %
Theoretisches Artikelvolumen gesamt: 23.78 cm^3
Beanspruchtes Box-Volumen: 1046.95 cm^3
Erreichte Fülldichte: 2.27 %
Teste Menge 25...
Spawne 25 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 29.83 mm
Relative Füllhöhe der Kiste: 21.31 %
Theoretisches Artikelvolumen gesamt: 59.45 cm^3
Beanspruchtes Box-Volumen: 1049.96 cm^3
Erreichte Fülldichte: 5.66 %
Teste Menge 50...
Spawne 50 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 29.76 mm
Relative Füllhöhe der Kiste: 21.26 %
Theoretisches Artikelvolumen gesamt: 118.91 cm^3
Beanspruchtes Box-Volumen: 1047.52 cm^3
Erreichte Fülldichte: 11.35 %
T

,quantity,fits_in_box,packing_density_percent,box_utilization_percent,filling_height_mm,articles_per_lhm
0,10,True,2.271451,None,29.742975,10.0
1,25,True,5.662381,None,29.828317,25.0
2,50,True,11.351100,None,29.759105,50.0
3,75,True,17.037441,None,29.740256,75.0
4,100,True,22.649484,None,29.828368,100.0


Teste Boxhöhe 80 mm...
Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 29.83 mm
Relative Füllhöhe der Kiste: 37.29 %
Theoretisches Artikelvolumen gesamt: 237.81 cm^3
Beanspruchtes Box-Volumen: 1049.97 cm^3
Erreichte Fülldichte: 22.65 %
Teste Boxhöhe 100 mm...
Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 29.79 mm
Relative Füllhöhe der Kiste: 29.79 %
Theoretisches Artikelvolumen gesamt: 237.81 cm^3
Beanspruchtes Box-Volumen: 1048.72 cm^3
Erreichte Fülldichte: 22.68 %
Teste Boxhöhe 120 mm...
Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 29.75 mm
Relative Füllhöhe der Kiste: 24.79 %
Theoretisches Artikelvolumen gesamt: 237.81 cm^3
Beanspruchtes Box-Volumen: 1047.31 cm^3
Erreichte Fülldichte: 22.71 %
Teste Boxhöhe 140 mm...
Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 29.83 mm
Relative Füllhöhe der Kiste: 21.31 %
Theo

,box_height_mm,fits_in_box,packing_density_percent,box_utilization_percent,filling_height_mm,articles_per_lhm
0,80,True,22.649139,None,29.828822,100.0
1,100,True,22.676206,None,29.793218,100.0
2,120,True,22.706789,None,29.753089,100.0
3,140,True,22.649484,None,29.828368,100.0
4,160,True,22.681213,None,29.786641,100.0
5,180,True,22.659396,None,29.815320,100.0
6,200,True,22.656880,None,29.818631,100.0
